In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio

In [ ]:
DIR_DATA = Path("data")
DIR_ICEYE_alined = DIR_DATA / "1-5_intermediate" / "1_ICEYE_aligned"
DIR_ICEYE_ZSCORE = DIR_DATA / "1-5_intermediate" / "2_ICEYE_Zscore"
DIR_ICEYE_ZSCORE_STATS = DIR_ICEYE_ZSCORE / "1_year_direction_side"

FILEPATH_ICEYE_LIST = DIR_DATA / "0_metadata" / "iceye-list-zscore.csv"

In [ ]:
def read_tiff(path):
    with rasterio.open(path) as src:
        image = src.read(1).astype(np.float32)
        profile = src.profile.copy()
    return image, profile


def save_tiff(path, image, profile):
    profile = profile.copy()
    profile.update(
        dtype="float32",
        count=1,
        compress="lzw"
    )
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(
            image.astype(np.float32),
            1
        )

In [ ]:
df = pd.read_csv(FILEPATH_ICEYE_LIST)

df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year

df["group"] = (
    df["year"].astype(str)
    + "-"
    + df["orbit_direction"]
    + "-"
    + df["look_side"]
)

# ----------------------------------------
# Generate Z-score maps
# ----------------------------------------
for group, group_df in df.groupby("group"):
    print(f"\n{group}")
    mean, profile = read_tiff(
        DIR_ICEYE_ZSCORE_STATS / f"{group}_mean.tif"
    )
    std, _ = read_tiff(
        DIR_ICEYE_ZSCORE_STATS / f"{group}_std.tif"
    )
    for _, row in group_df.iterrows():
        image_path = (
            DIR_ICEYE_alined
            / (
                Path(row["filename"]).stem
                + "_EPSG2958_res05m.tif"
            )
        )
        image, _ = read_tiff(image_path)
        zscore = (image - mean) / std
        zscore = np.clip(zscore, -5, 5)  # Optional: clip extreme values

        output_path = (
            DIR_ICEYE_ZSCORE
            / (
                Path(row["filename"]).stem
                + "_Zscore.tif"
            )
        )
        save_tiff(
            output_path,
            zscore,
            profile
        )

    print(f"Finished {group}")